# 🤗 x 🦾: Training ACT with LeRobot Notebook

Welcome to the **LeRobot ACT training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `ACT` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `ACT` policy for 100,000 steps typically takes **about 1.5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer.

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Setup
Install Conda (Miniforge) and configure your HuggingFace credentials. These are used throughout the notebook for dataset access, model uploads, and authentication.


In [1]:
!nvidia-smi

Thu Apr  9 08:50:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import subprocess, os
from getpass import getpass

# --- Install Miniforge (conda) without restarting the kernel ---
if not os.path.exists("/opt/conda/bin/conda"):
    print("Installing Miniforge...")
    subprocess.run(["wget", "-qO", "/tmp/miniforge.sh",
        "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"],
        check=True)
    subprocess.run(["bash", "/tmp/miniforge.sh", "-b", "-p", "/opt/conda"],
        check=True, stdout=subprocess.DEVNULL)
    print("Miniforge installed.")
else:
    print("Conda already installed.")

os.environ["PATH"] = "/opt/conda/bin:" + os.environ["PATH"]
print(subprocess.check_output(["conda", "--version"]).decode().strip())

# --- HuggingFace credentials ---
HF_USER = input("HuggingFace username: ").strip()
HF_TOKEN = getpass("HuggingFace token (hf_...): ")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_USER"] = HF_USER

!huggingface-cli login --token $HF_TOKEN

print(f"\nConfigured for user: {HF_USER}")

Installing Miniforge...
Miniforge installed.
conda 26.1.1
/bin/bash: line 1: huggingface-cli: command not found

Configured for user: cadenli


## Install LeRobot
This cell installs FFmpeg, clones the `lerobot` repository from Hugging Face, and installs the package in editable mode.


In [3]:
!git clone https://github.com/huggingface/lerobot.git
!conda install -y ffmpeg=7.1.1 -c conda-forge
!cd lerobot && pip install -e .

Cloning into 'lerobot'...
remote: Enumerating objects: 44744, done.
remote: Counting objects: 100% (416/416), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 44744 (delta 334), reused 282 (delta 260), pack-reused 44328 (from 2)
Receiving objects: 100% (44744/44744), 237.41 MiB | 38.88 MiB/s, done.
Resolving deltas: 100% (28329/28329), done.
Filtering content: 100% (45/45), 69.03 MiB | 15.58 MiB/s, done.
Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - ffmpeg=7.1.1


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    alsa-lib-1.2.15.3          |       hb03c661_0         571 KB  conda-forge
    aom-3.9.1                  |       hac33072_0         2.6 MB  conda-forge
    cairo-1.18.4               |       he90730b_1         966 KB  conda-

In [4]:
!python --version

Python 3.13.12


## Weights & Biases login
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging.

In [5]:
import os
from getpass import getpass

wandb_key = getpass("Paste your W&B API key: ")
os.environ["WANDB_API_KEY"] = wandb_key
!wandb login --relogin $WANDB_API_KEY
print("wandb configured.")

wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb configured.


## Merge Datasets

Uses LeRobot's built-in [`lerobot-edit-dataset`](https://huggingface.co/docs/lerobot/using_dataset_tools) tool to merge multiple recorded datasets into a single training dataset. All source datasets must share the same features (cameras, action space, etc.).

In [10]:
MERGED_REPO_ID = f"{HF_USER}/act-training-merged"

!cd lerobot && python src/lerobot/scripts/lerobot_edit_dataset.py \
  --new_repo_id={MERGED_REPO_ID} \
  --operation.type=merge \
  --operation.repo_ids="['{HF_USER}/act-training-02', '{HF_USER}/act-training-03', '{HF_USER}/act-training-04', '{HF_USER}/act-training-05', '{HF_USER}/act-training-06']" \
  --push_to_hub=true

INFO 2026-04-02 05:30:52 _dataset.py:393 Loading 5 datasets to merge
INFO 2026-04-02 05:30:52 eo_utils.py:108 Using video codec: libsvtav1
Fetching 4 files: 100% 4/4 [00:00<00:00,  5.51it/s]
Download complete: : 84.0kB [00:00, 114kB/s]              
Fetching 9 files: 100% 9/9 [00:00<00:00, 10.18it/s]7MB/s]                
Download complete: : 111MB [00:00, 137MB/s]              INFO 2026-04-02 05:30:54 eo_utils.py:108 Using video codec: libsvtav1


Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100% 4/4 [00:00<00:00,  7.69it/s]

Download complete: : 111MB [00:01, 61.0MB/s]              
Download complete: : 84.5kB [00:00, 156kB/s]
Fetching 9 files: 100% 9/9 [00:01<00:00,  8.48it/s]
Download complete: : 115MB [00:01, 161MB/s]              INFO 2026-04-02 05:30:56 eo_utils.py:108 Using video codec: libsvtav1


Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100% 4/4 [00:00<00:00,  7.99it/s]

Download complete: : 115MB [00:01, 67.3MB/s]              
Downloa

In [8]:
MERGED_REPO_ID="cadenli/hippo_combined"

## Start training ACT with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [configuration_act.py](../lerobot/common/policies/act/configuration_act.py), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this. Set to `False` if you do not plan on using Weights & Biases.

In [6]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!cd lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id={MERGED_REPO_ID} \
  --policy.type=act \
  --batch_size=64 \
  --output_dir=outputs/train/act_training_merged \
  --job_name=act_merged_training \
  --policy.device=cuda \
  --wandb.enable=True \
  --policy.repo_id={HF_USER}/act_train_hippo_1

INFO 2026-04-09 09:02:08 ot_train.py:197 {'batch_size': 64,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                           

## Auto-Upload Last Checkpoint

After training completes, the cell below automatically uploads the last checkpoint and training config to the Hugging Face Hub. The repo is created if it doesn't already exist.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo

OUTPUT_DIR = Path("lerobot/outputs/train/act_training_merged")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints" / "last" / "pretrained_model"
CONFIG_PATH = OUTPUT_DIR / "train_config.json"
POLICY_REPO_ID = f"{HF_USER}/act_train_hippo_1"

api = HfApi(token=os.environ.get("HF_TOKEN"))

create_repo(POLICY_REPO_ID, exist_ok=True, token=os.environ.get("HF_TOKEN"))
print(f"Target repo: {POLICY_REPO_ID}")

if CHECKPOINT_DIR.exists():
    print(f"\nUploading checkpoint from {CHECKPOINT_DIR}...")
    api.upload_folder(
        folder_path=str(CHECKPOINT_DIR),
        repo_id=POLICY_REPO_ID,
    )
    print("Checkpoint uploaded!")
else:
    print(f"\nWARNING: Checkpoint not found at {CHECKPOINT_DIR}")
    ckpt_base = OUTPUT_DIR / "checkpoints"
    if ckpt_base.exists():
        print("Available checkpoints:")
        for p in sorted(ckpt_base.iterdir()):
            print(f"  {p.name}")
    else:
        print(f"No checkpoints directory found at {ckpt_base}")

if CONFIG_PATH.exists():
    print(f"\nUploading {CONFIG_PATH.name}...")
    api.upload_file(
        path_or_fileobj=str(CONFIG_PATH),
        path_in_repo="train_config.json",
        repo_id=POLICY_REPO_ID,
    )
    print("Config uploaded!")

print(f"\nDone! https://huggingface.co/{POLICY_REPO_ID}")


### Verify Hugging Face Token

Confirms the HF_TOKEN environment variable was set correctly in the setup cell above.

In [ ]:
import os

if os.getenv("HF_TOKEN"):
    print("Hugging Face token loaded successfully.")
else:
    print("Error: HF_TOKEN not set. Re-run the Setup cell above.")

In [ ]:
!huggingface-cli whoami

### Download files from Hugging Face Hub

Downloads the training config and trained model files locally. This is optional.

In [ ]:
pass

### Verify `train_config.json` existence locally. This is needed for restarting training and testing of the policy. 

In [ ]:
!ls -l lerobot/outputs/train/act_training_merged/checkpoints/

In [ ]:
!huggingface-cli repo files {HF_USER}/act_train_hippo_1

In [ ]:
!huggingface-cli whoami

In [ ]:
pass

In [ ]:
!huggingface-cli whoami

### Create a new repository on Hugging Face Hub

This command will create a new repository under your Hugging Face account.

In [ ]:
pass

In [ ]:
pass